<a href="https://colab.research.google.com/github/Andrea-2215/BigData/blob/main/PART_2_Your_Data%2C_Our_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

Saving MobileGames.csv to MobileGames.csv


{'MobileGames.csv': b'Game,Release date,As of,Player count[a],Publisher(s),Ref.,Table_Number\r\nMobile Legends: Bang Bang,December 2017,January 2017,150million peakdaily players,Moontoon,,1\r\nPUBG Mobile,March 2018,August 2023,300 millionmonthly players,Tencent Games/Krafton,,1\r\nCall of Duty: Mobile,"October 1, 2019",Nov 2024,1billion downloads[b],Activision,,1\r\nAmong Us,"June 15, 2018",November 2020,485million[c],InnerSloth,,1\r\nMini World,"December 26, 2015",April 2020,400million,Minovate,,1\r\nDragon Ball Z: Dokkan Battle,"January 30, 2015",August 2021,350million,Bandai Namco Entertainment,,1\r\nSonic Dash,"March 7, 2013",February 2020,350million,Sega,,1\r\nHelix Jump,"February 10, 2018",December 2018,334million,Voodoo,,1\r\nGardenscapes: New Acres,August 2016,May 2020,324million,Playrix,,1\r\nHomescapes,August 2017,May 2020,312million,Playrix,,1\r\nSuper Mario Run,"December 15, 2016",August 2018,300million,Nintendo,,1\r\nTownship,"February 24, 2012",May 2020,274million,Playri

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
import re

# LAB ACTIVITY 4 - Your Data, Our Data. Part 2
# MEMBERS:
# BOMBALES, Andrea B.
# BORROMEO, Kathleen Ann B.
# SIOJO, Pauleen Anne D.

spark = SparkSession.builder \
    .appName("MobileGamesAnalysis_RDD") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# ─────────────────────────────────────────────────────────────
# MANUAL SCHEMA
# CSV columns: Game, Release date, As of, Player count[a],
#              Publisher(s), Ref., Table_Number
# We drop Ref. and Table_Number, and derive two extra columns.
# ─────────────────────────────────────────────────────────────
schema = StructType([
    StructField("Game",                  StringType(),  True),
    StructField("Release_Date",          StringType(),  True),
    StructField("As_Of",                 StringType(),  True),
    StructField("Player_Count_Raw",      StringType(),  True),
    StructField("Publisher",             StringType(),  True),
    StructField("Player_Count_Millions", DoubleType(),  True),
    StructField("Release_Year",          IntegerType(), True),
])

# ─────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────

def parse_player_count(raw):
    """
    Convert raw player-count string to a float (in millions).
    Handles:  '300 millionmonthly players'  -> 300.0
              '1billion downloads'           -> 1000.0
              '485million'                   -> 485.0
    """
    if not raw:
        return None
    text = raw.lower()
    m = re.search(r"([\d.]+)\s*billion", text)
    if m:
        return float(m.group(1)) * 1000.0
    m = re.search(r"([\d.]+)\s*million", text)
    if m:
        return float(m.group(1))
    m = re.search(r"([\d.]+)", text)
    if m:
        return float(m.group(1))
    return None


def parse_year(date_str):
    """Extract 4-digit year from any date string."""
    if not date_str:
        return None
    m = re.search(r"(\d{4})", date_str)
    return int(m.group(1)) if m else None


def parse_csv_line(line):
    """
    Split a CSV line respecting double-quoted fields that may contain commas.
    Returns a list of stripped strings.
    """
    fields = []
    current = []
    in_quotes = False
    for ch in line:
        if ch == '"':
            in_quotes = not in_quotes
        elif ch == ',' and not in_quotes:
            fields.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    fields.append(''.join(current).strip())
    return fields


# ─────────────────────────────────────────────────────────────
# LOAD CSV AS RDD
# ─────────────────────────────────────────────────────────────
raw_rdd = spark.sparkContext.textFile("MobileGames.csv")

# Pull header and detect column indices dynamically
header_line = raw_rdd.first()
headers = [h.strip() for h in parse_csv_line(header_line)]

IDX_GAME      = headers.index("Game")
IDX_REL_DATE  = headers.index("Release date")
IDX_AS_OF     = headers.index("As of")
IDX_PLAYERS   = headers.index("Player count[a]")
IDX_PUBLISHER = headers.index("Publisher(s)")

# ─────────────────────────────────────────────────────────────
# PARSE & CLEAN USING RDD OPERATIONS
# ─────────────────────────────────────────────────────────────

def build_row(line):
    """Map a raw CSV line -> Row (or None if invalid)."""
    fields = parse_csv_line(line)
    # Pad to avoid index errors
    while len(fields) <= max(IDX_GAME, IDX_REL_DATE, IDX_AS_OF, IDX_PLAYERS, IDX_PUBLISHER):
        fields.append("")

    game = fields[IDX_GAME].strip()
    if not game:
        return None   # skip rows missing a game name

    raw_date    = fields[IDX_REL_DATE].strip()  or None
    as_of       = fields[IDX_AS_OF].strip()      or None
    raw_players = fields[IDX_PLAYERS].strip()    or None
    publisher   = fields[IDX_PUBLISHER].strip()  or None

    return Row(
        Game                  = game,
        Release_Date          = raw_date,
        As_Of                 = as_of,
        Player_Count_Raw      = raw_players,
        Publisher             = publisher,
        Player_Count_Millions = parse_player_count(raw_players),
        Release_Year          = parse_year(raw_date),
    )


# Step 1 — drop header
data_rdd = raw_rdd.filter(lambda line: line != header_line)

# Step 2 — parse
parsed_rdd = data_rdd.map(build_row)

# Step 3 — remove None (missing game name / parse failure)
parsed_rdd = parsed_rdd.filter(lambda r: r is not None)

# Step 4 — remove duplicates via composite key
deduped_rdd = (
    parsed_rdd
    .map(lambda r: ((r.Game, r.Release_Date, r.Publisher, r.Player_Count_Raw), r))
    .reduceByKey(lambda a, b: a)
    .map(lambda kv: kv[1])
)

# ─────────────────────────────────────────────────────────────
# CONVERT RDD -> DATAFRAME (using the defined schema)
# ─────────────────────────────────────────────────────────────
df_clean = spark.createDataFrame(deduped_rdd, schema=schema)

print("=" * 60)
print("CLEANED DATAFRAME SAMPLE")
print("=" * 60)
df_clean.show(5, truncate=False)
print(f"Total records after cleaning: {df_clean.count()}")

# ─────────────────────────────────────────────────────────────
# INSIGHT 1 — Top 5 Games by Player Count
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("INSIGHT 1 — Top 5 Games by Player Count (Millions)")
print("=" * 60)

top5_rows = (
    deduped_rdd
    .filter(lambda r: r.Player_Count_Millions is not None)
    .map(lambda r: (r.Game, r.Publisher, r.Release_Year, r.Player_Count_Millions))
    .sortBy(lambda r: r[3], ascending=False)
    .take(5)
)

top5_games = spark.createDataFrame(
    top5_rows,
    schema=StructType([
        StructField("Game",                  StringType(),  True),
        StructField("Publisher",             StringType(),  True),
        StructField("Release_Year",          IntegerType(), True),
        StructField("Player_Count_Millions", DoubleType(),  True),
    ])
)
top5_games.show(truncate=False)

# ─────────────────────────────────────────────────────────────
# INSIGHT 2 — Publishers with the Most Titles
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("INSIGHT 2 — Publishers with the Most Games in Dataset")
print("=" * 60)

publisher_counts = (
    deduped_rdd
    .filter(lambda r: r.Publisher is not None)
    .map(lambda r: (r.Publisher, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda r: r[1], ascending=False)
)

publisher_titles = spark.createDataFrame(
    publisher_counts,
    schema=StructType([
        StructField("Publisher",    StringType(),  True),
        StructField("Total_Titles", IntegerType(), True),
    ])
)
publisher_titles.show(10, truncate=False)

# ─────────────────────────────────────────────────────────────
# INSIGHT 3 — Most Active Release Years
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("INSIGHT 3 — Most Active Release Years (by number of games)")
print("=" * 60)

year_counts = (
    deduped_rdd
    .filter(lambda r: r.Release_Year is not None)
    .map(lambda r: (r.Release_Year, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda r: r[1], ascending=False)
)

release_year_counts = spark.createDataFrame(
    year_counts,
    schema=StructType([
        StructField("Release_Year",   IntegerType(), True),
        StructField("Games_Released", IntegerType(), True),
    ])
)
release_year_counts.show(10)

# ─────────────────────────────────────────────────────────────
# WRITE RESULTS TO PARQUET
# ─────────────────────────────────────────────────────────────
df_clean           .write.mode("overwrite").parquet("clean_mobile_games.parquet")
top5_games         .write.mode("overwrite").parquet("top5_games.parquet")
publisher_titles   .write.mode("overwrite").parquet("publisher_titles.parquet")
release_year_counts.write.mode("overwrite").parquet("release_year_counts.parquet")

print("=" * 60)
print("Parquet files saved:")
print("  - clean_mobile_games.parquet")
print("  - top5_games.parquet")
print("  - publisher_titles.parquet")
print("  - release_year_counts.parquet")
print("=" * 60)

spark.stop()

CLEANED DATAFRAME SAMPLE
+--------------------+-----------------+--------------+---------------------+-------------------+---------------------+------------+
|Game                |Release_Date     |As_Of         |Player_Count_Raw     |Publisher          |Player_Count_Millions|Release_Year|
+--------------------+-----------------+--------------+---------------------+-------------------+---------------------+------------+
|Call of Duty: Mobile|October 1, 2019  |Nov 2024      |1billion downloads[b]|Activision         |1000.0               |2019        |
|Sonic Dash          |March 7, 2013    |February 2020 |350million           |Sega               |350.0                |2013        |
|Township            |February 24, 2012|May 2020      |274million           |Playrix            |274.0                |2012        |
|Knives Out          |November 2017    |September 2018|250million           |NetEase            |250.0                |2017        |
|Angry Birds 2       |July 30, 2015    |Dece

In [ ]:
# Import SparkSession to start a Spark application
from pyspark.sql import SparkSession

# Import data types used to define the structure (schema) of the dataset
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Import Spark SQL functions used for data cleaning, transformation, and analysis
from pyspark.sql.functions import (
    col, regexp_replace, regexp_extract, trim,   # column operations and regex extraction
    count, sum as spark_sum, desc, year, to_date # aggregation and sorting functions
)

# LAB TITLE
#LAB ACTIVITY 4 - Your Data, Our Data. Part 2

# Group members who worked on the activity
#MEMBERS:
#BOMBALES, Andrea B.
#BORROMEO, Kathleen Ann B.
#SIOJO, Pauleen Anne D.

# Create a Spark session (entry point for Spark DataFrame operations)
spark = SparkSession.builder \
    .appName("MobileGamesAnalysis") \  # Name of the Spark application
    .getOrCreate()                    # Start or retrieve the Spark session

# Reduce Spark log messages so only errors appear
spark.sparkContext.setLogLevel("ERROR")

# ---------------------------
# DEFINE DATASET STRUCTURE
# ---------------------------

# Define the schema (structure) of the CSV dataset
schema = StructType([
    StructField("Game",             StringType(),  True),   # Game name column
    StructField("Release_Date",     StringType(),  True),   # Release date of the game
    StructField("As_Of",            StringType(),  True),   # Date when player count was recorded
    StructField("Player_Count_Raw", StringType(),  True),   # Raw player count (e.g., "1 billion", "50 million")
    StructField("Publisher",        StringType(),  True),   # Game publisher
    StructField("Ref",              StringType(),  True),   # Reference source column
    StructField("Table_Number",     IntegerType(), True),   # Table number in the dataset
])

# ---------------------------
# LOAD CSV DATASET
# ---------------------------

# Read the CSV file using the defined schema
df = spark.read.csv(
    "MobileGames.csv",  # CSV file path
    header=True,        # First row contains column names
    schema=schema       # Apply predefined schema
)

# ---------------------------
# DATA CLEANING
# ---------------------------

# Remove unnecessary columns that are not needed for analysis
df_clean = df.drop("Ref", "Table_Number")

# Create a new column converting player counts expressed in billions to millions
df_clean = df_clean.withColumn(
    "Player_Count_Millions",
    # Extract number before the word "billion" then multiply by 1000 to convert to millions
    regexp_extract(col("Player_Count_Raw"), r"([\d.]+)\s*billion", 1).cast("double") * 1000
).withColumn(
    # If the previous step produced null (not billion), extract the value in millions
    "Player_Count_Millions",
    col("Player_Count_Millions").isNotNull().cast("double") *
        col("Player_Count_Millions") +
    col("Player_Count_Millions").isNull().cast("double") *
        regexp_extract(col("Player_Count_Raw"), r"([\d.]+)\s*million", 1).cast("double")
)

# Import the when() function for conditional logic
from pyspark.sql.functions import when

# Cleaner approach using conditional logic (CASE statement style)
df_clean = df_clean.withColumn(
    "Player_Count_Millions",
    when(
        col("Player_Count_Raw").rlike("(?i)billion"),  # Check if value contains "billion"
        regexp_extract(col("Player_Count_Raw"), r"([\d.]+)", 1).cast("double") * 1000
    ).otherwise(
        regexp_extract(col("Player_Count_Raw"), r"([\d.]+)", 1).cast("double") # Otherwise treat as million
    )
)

# Extract the release year from the Release_Date column
df_clean = df_clean.withColumn(
    "Release_Year",
    regexp_extract(col("Release_Date"), r"(\d{4})", 1).cast("integer") # Extract 4-digit year
)

# Remove extra spaces in text columns
df_clean = df_clean \
    .withColumn("Game",      trim(col("Game"))) \       # Clean game names
    .withColumn("Publisher", trim(col("Publisher")))   # Clean publisher names

# Remove rows where the Game name is missing
df_clean = df_clean.dropna(subset=["Game"])

# Remove duplicate records from the dataset
df_clean = df_clean.dropDuplicates()

# ---------------------------
# INSIGHT 1
# ---------------------------

# Display section header
print("=" * 60)
print("INSIGHT 1 — Top 5 Games by Player Count (Millions)")
print("=" * 60)

# Find the top 5 games with the highest player counts
top5_games = df_clean \
    .filter(col("Player_Count_Millions").isNotNull()) \     # Remove rows without player counts
    .select("Game", "Publisher", "Release_Year", "Player_Count_Millions") \ # Select important columns
    .orderBy(desc("Player_Count_Millions")) \               # Sort by highest player count
    .limit(5)                                               # Keep only top 5 results

# Show the result table
top5_games.show(truncate=False)

# ---------------------------
# INSIGHT 2
# ---------------------------

print("=" * 60)
print("INSIGHT 2 — Publishers with the Most Games in Dataset")
print("=" * 60)

# Count how many games each publisher has
publisher_titles = df_clean \
    .groupBy("Publisher") \                        # Group rows by publisher
    .agg(count("Game").alias("Total_Titles")) \    # Count number of games
    .orderBy(desc("Total_Titles"))                 # Sort from highest to lowest

# Show top 10 publishers
publisher_titles.show(10, truncate=False)

# ---------------------------
# INSIGHT 3
# ---------------------------

print("=" * 60)
print("INSIGHT 3 — Most Active Release Years (by number of games)")
print("=" * 60)

# Count how many games were released per year
release_year_counts = df_clean \
    .filter(col("Release_Year").isNotNull()) \     # Remove rows without year
    .groupBy("Release_Year") \                     # Group by year
    .agg(count("Game").alias("Games_Released")) \  # Count number of games
    .orderBy(desc("Games_Released"))               # Sort by most games released

# Show the top 10 years
release_year_counts.show(10)

# ---------------------------
# SAVE RESULTS
# ---------------------------

# Save cleaned dataset and analysis results in Parquet format (efficient big data storage)
df_clean          .write.mode("overwrite").parquet("clean_mobile_games.parquet")
top5_games        .write.mode("overwrite").parquet("top5_games.parquet")
publisher_titles  .write.mode("overwrite").parquet("publisher_titles.parquet")
release_year_counts.write.mode("overwrite").parquet("release_year_counts.parquet")

# Print confirmation that files were saved
print("=" * 60)
print("Parquet files saved:")
print("  -clean_mobile_games.parquet")
print("  -top5_games.parquet")
print("  -publisher_titles.parquet")
print("  -release_year_counts.parquet")
print("=" * 60)

# Stop the Spark session to free resources
spark.stop()

SyntaxError: unexpected character after line continuation character (3800298741.py, line 26)